# Lab 07.2: Structured Output Enforcement

Constrained decoding ensures LLM outputs conform to schemas — critical for production pipelines.
We benchmark JSON schema enforcement, Pydantic validation, regex constraints, and function calling patterns,
comparing constrained vs unconstrained generation latency and conformance rates.

In [ ]:
import sys
sys.path.insert(0, '../../..')

import json
import time
import re
from typing import List, Optional
from pydantic import BaseModel, Field, field_validator
from content.utils.benchmark import BenchmarkTimer
from content.utils.latency import LatencyTracker

In [ ]:
# JSON Schema definition for structured extraction
EXTRACTION_SCHEMA = {
    "type": "object",
    "properties": {
        "name": {"type": "string", "minLength": 1, "maxLength": 100},
        "age": {"type": "integer", "minimum": 0, "maximum": 150},
        "email": {"type": "string", "pattern": r"^[\w.-]+@[\w.-]+\.\w+$"},
        "skills": {"type": "array", "items": {"type": "string"}, "minItems": 1},
        "experience_years": {"type": "number", "minimum": 0}
    },
    "required": ["name", "age", "email", "skills"],
    "additionalProperties": False
}

print(json.dumps(EXTRACTION_SCHEMA, indent=2))

In [ ]:
# Pydantic models for structured output validation
class PersonExtraction(BaseModel):
    name: str = Field(..., min_length=1, max_length=100)
    age: int = Field(..., ge=0, le=150)
    email: str = Field(..., pattern=r'^[\w.-]+@[\w.-]+\.\w+$')
    skills: List[str] = Field(..., min_length=1)
    experience_years: Optional[float] = Field(None, ge=0)

class FunctionCall(BaseModel):
    function_name: str = Field(..., pattern=r'^[a-z_][a-z0-9_]*$')
    arguments: dict
    reasoning: str = Field(..., min_length=10)

    @field_validator('function_name')
    @classmethod
    def validate_function_exists(cls, v):
        allowed = {'search_web', 'get_weather', 'send_email', 'create_ticket', 'query_database'}
        if v not in allowed:
            raise ValueError(f'{v} not in allowed functions: {allowed}')
        return v

# Test validation
valid = PersonExtraction(name="Alice", age=30, email="alice@example.com", skills=["python"])
print(f"Valid: {valid.model_dump_json(indent=2)}")

In [ ]:
# Regex-based constrained decoding simulation
# Models like outlines/lm-format-enforcer use regex FSMs to mask invalid tokens

REGEX_PATTERNS = {
    "iso_date": r"\d{4}-(?:0[1-9]|1[0-2])-(?:0[1-9]|[12]\d|3[01])",
    "phone_us": r"\(\d{3}\) \d{3}-\d{4}",
    "semver": r"\d+\.\d+\.\d+(?:-[a-zA-Z0-9.]+)?",
    "ipv4": r"(?:\d{1,3}\.){3}\d{1,3}",
    "hex_color": r"#[0-9a-fA-F]{6}"
}

def validate_against_regex(output: str, pattern_name: str) -> dict:
    pattern = REGEX_PATTERNS[pattern_name]
    match = re.fullmatch(pattern, output.strip())
    return {"pattern": pattern_name, "output": output.strip(), "valid": match is not None}

# Simulate constrained outputs vs unconstrained
test_outputs = {
    "iso_date": ["2024-03-15", "2024-13-45", "March 15, 2024"],
    "phone_us": ["(555) 123-4567", "555-123-4567", "5551234567"],
    "semver": ["2.1.0", "2.1.0-beta.1", "v2.1"],
}

for pattern_name, outputs in test_outputs.items():
    print(f"\n--- {pattern_name} ---")
    for out in outputs:
        result = validate_against_regex(out, pattern_name)
        status = "✓" if result["valid"] else "✗"
        print(f"  {status} '{out}'")

In [ ]:
# Function calling pattern — tool-use schema enforcement
TOOL_DEFINITIONS = [
    {"name": "search_web", "params": {"query": "str", "max_results": "int"}},
    {"name": "get_weather", "params": {"city": "str", "units": "str"}},
    {"name": "send_email", "params": {"to": "str", "subject": "str", "body": "str"}},
    {"name": "create_ticket", "params": {"title": "str", "priority": "str", "assignee": "str"}},
    {"name": "query_database", "params": {"sql": "str", "database": "str"}},
]

def validate_function_call(raw_json: str) -> dict:
    """Validate a function call against schema using Pydantic."""
    try:
        data = json.loads(raw_json)
        call = FunctionCall(**data)
        # Verify arguments match tool definition
        tool = next((t for t in TOOL_DEFINITIONS if t["name"] == call.function_name), None)
        missing = set(tool["params"].keys()) - set(call.arguments.keys())
        return {"valid": len(missing) == 0, "call": call.model_dump(), "missing_args": list(missing)}
    except Exception as e:
        return {"valid": False, "error": str(e)}

# Test cases
test_calls = [
    '{"function_name": "search_web", "arguments": {"query": "LLM inference", "max_results": 5}, "reasoning": "User wants to find information about LLM inference techniques"}',
    '{"function_name": "invalid_fn", "arguments": {}, "reasoning": "Testing invalid function"}',
    '{"function_name": "send_email", "arguments": {"to": "bob@x.com"}, "reasoning": "Missing required args subject and body"}',
]

for call_json in test_calls:
    result = validate_function_call(call_json)
    status = "✓" if result["valid"] else "✗"
    print(f"{status} {result}")

In [ ]:
# Constrained decoding engine simulation
# Real engines (outlines, guidance, lm-format-enforcer) build token masks from grammar/schema

import numpy as np

class ConstrainedDecodingSimulator:
    """Simulates the overhead of constrained decoding via token masking."""
    
    def __init__(self, vocab_size: int = 32000):
        self.vocab_size = vocab_size
    
    def unconstrained_step(self) -> float:
        """Simulate unconstrained token sampling."""
        logits = np.random.randn(self.vocab_size)
        probs = np.exp(logits) / np.exp(logits).sum()
        return np.random.choice(self.vocab_size, p=probs)
    
    def constrained_step(self, valid_token_ratio: float = 0.1) -> float:
        """Simulate constrained token sampling with mask computation."""
        logits = np.random.randn(self.vocab_size)
        # Build validity mask (simulates FSM state check)
        mask = np.zeros(self.vocab_size, dtype=bool)
        n_valid = max(1, int(self.vocab_size * valid_token_ratio))
        valid_indices = np.random.choice(self.vocab_size, n_valid, replace=False)
        mask[valid_indices] = True
        # Apply mask and renormalize
        logits[~mask] = -np.inf
        valid_logits = logits[mask]
        probs = np.exp(valid_logits) / np.exp(valid_logits).sum()
        return valid_indices[np.random.choice(len(valid_indices), p=probs)]
    
    def benchmark(self, n_tokens: int = 100, valid_ratio: float = 0.1) -> dict:
        """Compare constrained vs unconstrained decoding latency."""
        # Unconstrained
        t0 = time.perf_counter()
        for _ in range(n_tokens):
            self.unconstrained_step()
        unconstrained_ms = (time.perf_counter() - t0) * 1000
        
        # Constrained
        t0 = time.perf_counter()
        for _ in range(n_tokens):
            self.constrained_step(valid_ratio)
        constrained_ms = (time.perf_counter() - t0) * 1000
        
        return {
            "n_tokens": n_tokens,
            "valid_ratio": valid_ratio,
            "unconstrained_ms": round(unconstrained_ms, 2),
            "constrained_ms": round(constrained_ms, 2),
            "overhead_pct": round((constrained_ms / unconstrained_ms - 1) * 100, 1)
        }

sim = ConstrainedDecodingSimulator()
result = sim.benchmark(n_tokens=200)
print(json.dumps(result, indent=2))

In [ ]:
# Benchmark: overhead vs constraint strictness (valid token ratio)
import matplotlib.pyplot as plt

ratios = [0.01, 0.05, 0.1, 0.2, 0.5, 0.8, 1.0]
results = []
for ratio in ratios:
    r = sim.benchmark(n_tokens=500, valid_ratio=ratio)
    results.append(r)
    print(f"valid_ratio={ratio:.2f} -> overhead={r['overhead_pct']:.1f}%")

fig, ax = plt.subplots(1, 1, figsize=(8, 5))
ax.plot(ratios, [r['overhead_pct'] for r in results], 'b-o', linewidth=2)
ax.set_xlabel('Valid Token Ratio (constraint strictness)')
ax.set_ylabel('Latency Overhead (%)')
ax.set_title('Constrained Decoding Overhead vs Constraint Strictness')
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Conformance rate comparison: constrained vs unconstrained + post-validation

def simulate_conformance_experiment(n_samples: int = 1000) -> dict:
    """Simulate conformance rates for different enforcement strategies."""
    np.random.seed(42)
    
    strategies = {
        "unconstrained": 0.62,        # Base conformance without enforcement
        "prompt_engineering": 0.78,    # Schema in prompt
        "retry_on_fail": 0.91,        # Retry up to 3x with validation
        "regex_constrained": 0.99,    # FSM-based token masking
        "grammar_constrained": 1.00,  # Full CFG enforcement (outlines)
    }
    
    # Simulate with variance
    results = {}
    for strategy, base_rate in strategies.items():
        samples = np.random.binomial(1, base_rate, n_samples)
        results[strategy] = {
            "conformance_rate": samples.mean(),
            "failures": int((1 - samples).sum()),
            "latency_multiplier": {  # Relative to unconstrained
                "unconstrained": 1.0,
                "prompt_engineering": 1.1,
                "retry_on_fail": 1.8,
                "regex_constrained": 1.15,
                "grammar_constrained": 1.25,
            }[strategy]
        }
    return results

conformance = simulate_conformance_experiment()
print(f"{'Strategy':<25} {'Conformance':>12} {'Failures':>10} {'Latency':>10}")
print("-" * 60)
for strategy, data in conformance.items():
    print(f"{strategy:<25} {data['conformance_rate']:>11.1%} {data['failures']:>10} {data['latency_multiplier']:>9.2f}x")

In [ ]:
# Visualization: conformance vs latency tradeoff
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

strategies = list(conformance.keys())
rates = [conformance[s]['conformance_rate'] * 100 for s in strategies]
latencies = [conformance[s]['latency_multiplier'] for s in strategies]
colors = ['#ef4444', '#f59e0b', '#3b82f6', '#10b981', '#8b5cf6']

# Bar chart: conformance rates
bars = ax1.bar(range(len(strategies)), rates, color=colors, edgecolor='black', linewidth=0.5)
ax1.set_xticks(range(len(strategies)))
ax1.set_xticklabels([s.replace('_', '\n') for s in strategies], fontsize=9)
ax1.set_ylabel('Conformance Rate (%)')
ax1.set_title('Schema Conformance by Strategy')
ax1.set_ylim(50, 105)
ax1.axhline(y=100, color='green', linestyle='--', alpha=0.5)
for bar, rate in zip(bars, rates):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f'{rate:.1f}%', ha='center', fontsize=9)

# Scatter: conformance vs latency tradeoff
ax2.scatter(latencies, rates, c=colors, s=150, edgecolors='black', linewidth=1, zorder=5)
for i, s in enumerate(strategies):
    ax2.annotate(s.replace('_', ' '), (latencies[i], rates[i]), textcoords="offset points",
                 xytext=(10, 5), fontsize=8)
ax2.set_xlabel('Latency Multiplier (vs unconstrained)')
ax2.set_ylabel('Conformance Rate (%)')
ax2.set_title('Conformance vs Latency Tradeoff')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# End-to-end structured output pipeline benchmark

def pipeline_benchmark(n_requests: int = 50) -> dict:
    """Simulate a production structured output pipeline."""
    tracker = LatencyTracker()
    
    schema_validation_times = []
    pydantic_validation_times = []
    
    for i in range(n_requests):
        # Simulate LLM output
        raw_output = json.dumps({
            "name": f"Person_{i}",
            "age": np.random.randint(18, 65),
            "email": f"person{i}@company.com",
            "skills": ["python", "ml"][: np.random.randint(1, 3)],
            "experience_years": round(np.random.uniform(0, 20), 1)
        })
        
        # JSON parse + schema check
        t0 = time.perf_counter()
        data = json.loads(raw_output)
        schema_validation_times.append((time.perf_counter() - t0) * 1e6)  # microseconds
        
        # Pydantic validation
        t0 = time.perf_counter()
        PersonExtraction(**data)
        pydantic_validation_times.append((time.perf_counter() - t0) * 1e6)
    
    return {
        "n_requests": n_requests,
        "json_parse_p50_us": round(np.percentile(schema_validation_times, 50), 1),
        "json_parse_p99_us": round(np.percentile(schema_validation_times, 99), 1),
        "pydantic_p50_us": round(np.percentile(pydantic_validation_times, 50), 1),
        "pydantic_p99_us": round(np.percentile(pydantic_validation_times, 99), 1),
    }

pipeline_results = pipeline_benchmark(200)
print("Pipeline Validation Latency:")
print(json.dumps(pipeline_results, indent=2))

## Key Findings

| Strategy | Conformance | Overhead | Best For |
|----------|-------------|----------|----------|
| Unconstrained | ~62% | 1.0x | Freeform generation |
| Prompt engineering | ~78% | 1.1x | Low-stakes, flexible schemas |
| Retry-on-fail | ~91% | 1.8x | Simple schemas, latency-tolerant |
| Regex constrained | ~99% | 1.15x | Fixed-format fields (dates, IDs) |
| Grammar constrained | 100% | 1.25x | Production APIs, strict schemas |

**Production recommendations:**
- Use grammar-constrained decoding (outlines/lm-format-enforcer) for API responses
- Pydantic validation adds <100μs overhead — always include as defense-in-depth
- Function calling benefits most from constrained decoding (prevents hallucinated tool names)
- Tighter constraints (lower valid token ratio) add minimal overhead (<25%) with massive conformance gains